# Speaker RVC POC — XTTS timbre correction

Train a supervised RVC v2 converter on the frozen 24-minute experiment bundle, then compare identical raw XTTS probes across checkpoints and retrieval rates. This is private research audio: do not publish it or represent it as a genuine statement by the target speaker.

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re, shutil, subprocess, sys, zipfile

RUN_FULL_DATASET = False
MODEL_NAME = "speaker_rvc_poc_v1" if not RUN_FULL_DATASET else "speaker_rvc_full_v1"
BUNDLE_NAME = "speaker_rvc_poc_24m.zip" if not RUN_FULL_DATASET else "speaker_rvc_full_74m.zip"
DRIVE_ROOT = Path("/content/drive/MyDrive/speaker_rvc")
BUNDLE_PATH = DRIVE_ROOT / BUNDLE_NAME
DRIVE_OUT = DRIVE_ROOT / MODEL_NAME
APPLIO = Path("/content/Applio")
WORK = Path("/content/speaker_rvc_work")
BUNDLE_ROOT = WORK / "bundle"
COMPARISON_DIR = DRIVE_OUT / "comparisons" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

APPLIO_COMMIT = "3e5e248c2bd003f65f3e128dd12677533def27be"
SAMPLE_RATE = 32000
TOTAL_EPOCHS = 200 if not RUN_FULL_DATASET else 250
BATCH_SIZE = 8
SAVE_EVERY_EPOCH = 25
F0_METHOD = "rmvpe"
EMBEDDER = "contentvec"
VOCODER = "HiFi-GAN"
AUDITION_EPOCHS = (100, 150, 200)
CHOSEN_EPOCH = 150
INDEX_RATES = (0.25, 0.50, 0.75)

assert not RUN_FULL_DATASET, "POC first: review the five-probe gate before enabling full training"
print({"model": MODEL_NAME, "bundle": str(BUNDLE_PATH), "epochs": TOTAL_EPOCHS, "batch": BATCH_SIZE})

## 2. Mount Drive and verify GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. Select Runtime > Change runtime type > GPU.")
gpu = torch.cuda.get_device_properties(0)
free_gb = shutil.disk_usage("/content").free / 1e9
print("Python", sys.version)
print("Torch", torch.__version__, "CUDA", torch.version.cuda)
print("GPU", gpu.name, "VRAM_GB", round(gpu.total_memory / 1e9, 2), "disk_free_GB", round(free_gb, 1))
if free_gb < 25:
    raise RuntimeError(f"At least 25 GB ephemeral disk is required; only {free_gb:.1f} GB is free")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

## 3. Install pinned Applio

In [ ]:
def run(command, *, cwd=None):
    printable = " ".join(str(item) for item in command)
    print("+", printable, flush=True)
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)

if not APPLIO.exists():
    run(["git", "clone", "--filter=blob:none", "https://github.com/IAHispano/Applio.git", APPLIO])
run(["git", "-C", APPLIO, "fetch", "--tags", "--force"])
run(["git", "-C", APPLIO, "checkout", "--detach", APPLIO_COMMIT])
# Audit equivalent: git rev-parse HEAD
actual_commit = subprocess.check_output(
    ["git", "-C", APPLIO, "rev-parse", "HEAD"], text=True
).strip()
assert actual_commit == APPLIO_COMMIT, (actual_commit, APPLIO_COMMIT)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
run([
    "uv", "pip", "install", "--system", "-q", "-r", "requirements.txt",
    "--extra-index-url", "https://download.pytorch.org/whl/cu128",
    "--index-strategy", "unsafe-best-match",
], cwd=APPLIO)
run([
    sys.executable, "core.py", "prerequisites",
    "--models", "True", "--pretraineds_hifigan", "True",
], cwd=APPLIO)
print("Pinned Applio ready:", actual_commit)

## 4. Extract and verify experiment bundle

In [ ]:
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Upload the experiment bundle to {BUNDLE_PATH}")
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
BUNDLE_ROOT.mkdir(parents=True)

with zipfile.ZipFile(BUNDLE_PATH) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Corrupt ZIP member: {bad_member}")
    archive.extractall(BUNDLE_ROOT)

manifest_path = BUNDLE_ROOT / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["schema"] == "rvc_poc_bundle.v1", manifest.get("schema")
for section in ("training", "eval_refs", "probes"):
    for row in manifest[section]:
        path = BUNDLE_ROOT / row["path"]
        if not path.is_file():
            raise FileNotFoundError(path)
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        if actual != row["sha256"]:
            raise RuntimeError(f"manifest hash mismatch: {row['path']}")
if not RUN_FULL_DATASET:
    assert 1440.0 <= manifest["training_duration_s"] <= 1451.1
    assert len(manifest["eval_refs"]) == 8
    assert len(manifest["probes"]) == 5
print("Bundle verified:", len(manifest["training"]), "clips", round(manifest["training_duration_s"] / 60, 2), "min")

## 5. Bind resumable logs to Drive

In [ ]:
logs_root = APPLIO / "logs"
logs_root.mkdir(parents=True, exist_ok=True)
drive_log_dir = DRIVE_OUT / "logs" / MODEL_NAME
drive_log_dir.mkdir(parents=True, exist_ok=True)
local_log_dir = logs_root / MODEL_NAME
if local_log_dir.is_symlink():
    local_log_dir.unlink()
elif local_log_dir.exists():
    shutil.copytree(local_log_dir, drive_log_dir, dirs_exist_ok=True)
    shutil.rmtree(local_log_dir)
os.symlink(str(drive_log_dir), str(local_log_dir), target_is_directory=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
print("Resumable log directory:", drive_log_dir)

## 6. Preprocess curated audio

In [ ]:
preprocess_marker = drive_log_dir / ".preprocess_complete"
if not preprocess_marker.exists():
    run([
        sys.executable, "core.py", "preprocess",
        "--model_name", MODEL_NAME,
        "--dataset_path", str(BUNDLE_ROOT / "dataset"),
        "--sample_rate", str(SAMPLE_RATE),
        "--cpu_cores", str(os.cpu_count() or 2),
        "--cut_preprocess", "Skip",
        "--process_effects", "False",
        "--noise_reduction", "False",
        "--noise_reduction_strength", "0.7",
        "--chunk_len", "3.0",
        "--overlap_len", "0.3",
        "--normalization_mode", "none",
    ], cwd=APPLIO)
    preprocess_marker.write_text("complete\n", encoding="utf-8")
else:
    print("Preprocess already complete; using Drive state")

## 7. Extract RMVPE and ContentVec features

In [ ]:
extract_marker = drive_log_dir / ".extract_complete"
if not extract_marker.exists():
    run([
        sys.executable, "core.py", "extract",
        "--model_name", MODEL_NAME,
        "--f0_method", F0_METHOD,
        "--sample_rate", str(SAMPLE_RATE),
        "--cpu_cores", str(os.cpu_count() or 2),
        "--gpu", "0",
        "--embedder_model", EMBEDDER,
        "--embedder_model_custom", "",
        "--include_mutes", "2",
    ], cwd=APPLIO)
    extract_marker.write_text("complete\n", encoding="utf-8")
else:
    print("Feature extraction already complete; using Drive state")

## 8. Train RVC v2 and build index

In [ ]:
train_marker = drive_log_dir / f".trained_{TOTAL_EPOCHS}"
if not train_marker.exists():
    run([
        sys.executable, "core.py", "train",
        "--model_name", MODEL_NAME,
        "--save_every_epoch", str(SAVE_EVERY_EPOCH),
        "--save_only_latest", "False",
        "--save_every_weights", "True",
        "--total_epoch", str(TOTAL_EPOCHS),
        "--sample_rate", str(SAMPLE_RATE),
        "--batch_size", str(BATCH_SIZE),
        "--gpu", "0",
        "--pretrained", "True",
        "--custom_pretrained", "False",
        "--overtraining_detector", "False",
        "--cleanup", "False",
        "--cache_data_in_gpu", "False",
        "--vocoder", VOCODER,
        "--checkpointing", "False",
    ], cwd=APPLIO)
    train_marker.write_text("complete\n", encoding="utf-8")
else:
    print("Requested training ceiling already completed")

## 9. Verify and export artifacts

In [ ]:
all_weights = sorted(drive_log_dir.rglob("*.pth"))
weights = [
    path for path in all_weights
    if not path.name.startswith(("G_", "D_"))
]
indexes = sorted(drive_log_dir.rglob("*.index"))
if not weights:
    raise RuntimeError(f"No RVC inference weights found under {drive_log_dir}")
if not indexes:
    raise RuntimeError(f"No FAISS index found under {drive_log_dir}")
index_path = max(indexes, key=lambda path: path.stat().st_mtime)
artifact_dir = DRIVE_OUT / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
for path in [*weights, index_path]:
    shutil.copy2(path, artifact_dir / path.name)
print("weights:", [path.name for path in weights])
print("index:", index_path.name)
print("Training checkpoints remain in Drive logs for resume; results export contains inference weights only")

## 10. Audition epochs 100, 150, and 200

In [ ]:
from IPython.display import Audio, display

def epoch_number(path):
    matches = re.findall(r"(?:_|-)(\d+)e(?:_|-|\.)", path.name)
    return int(matches[-1]) if matches else None

epoch_weights = {epoch_number(path): path for path in weights if epoch_number(path) is not None}

def require_epoch(epoch):
    if epoch not in epoch_weights:
        available = sorted(value for value in epoch_weights if value is not None)
        raise RuntimeError(f"Requested epoch {epoch} is missing; available epochs: {available}")
    return epoch_weights[epoch]

def infer(source_path, output_path, weight_path, index_rate):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    run([
        sys.executable, "core.py", "infer",
        "--pitch", "0",
        "--volume_envelope", "0.8",
        "--index_rate", str(index_rate),
        "--protect", "0.5",
        "--f0_autotune", "False",
        "--f0_method", F0_METHOD,
        "--input_path", str(source_path),
        "--output_path", str(output_path),
        "--pth_path", str(weight_path),
        "--index_path", str(index_path),
        "--split_audio", "False",
        "--clean_audio", "False",
        "--clean_strength", "0.7",
        "--export_format", "WAV",
        "--embedder_model", EMBEDDER,
        "--embedder_model_custom", "",
        "--formant_shifting", "False",
        "--formant_qfrency", "1.0",
        "--formant_timbre", "1.0",
        "--post_process", "False",
    ], cwd=APPLIO)
    if not output_path.is_file():
        raise RuntimeError(f"Inference did not create {output_path}")

probe_one = BUNDLE_ROOT / manifest["probes"][0]["path"]
display(Audio(str(probe_one)), "raw XTTS")
audition_rows = []
for epoch in AUDITION_EPOCHS:
    weight = require_epoch(epoch)
    output = COMPARISON_DIR / f"probe_01_e{epoch}_i050.wav"
    infer(probe_one, output, weight, 0.50)
    audition_rows.append({"probe_id": "probe_01", "epoch": epoch, "index_rate": 0.50, "path": str(output)})
    print("epoch", epoch)
    display(Audio(str(output)))

## 11. Choose epoch and sweep index rates

In [ ]:
# After listening above, edit CHOSEN_EPOCH in Configuration or here and rerun this cell.
chosen_weight = require_epoch(CHOSEN_EPOCH)
conversion_rows = []
for probe in manifest["probes"]:
    source = BUNDLE_ROOT / probe["path"]
    conversion_rows.append({
        "kind": "raw", "probe_id": probe["id"], "text": probe["text"],
        "path": str(source), "epoch": None, "index_rate": None,
    })
    for rate in INDEX_RATES:
        rate_tag = f"{round(rate * 100):03d}"
        output = COMPARISON_DIR / f"{probe['id']}_e{CHOSEN_EPOCH}_i{rate_tag}.wav"
        infer(source, output, chosen_weight, rate)
        conversion_rows.append({
            "kind": "converted", "probe_id": probe["id"], "text": probe["text"],
            "path": str(output), "epoch": CHOSEN_EPOCH, "index_rate": rate,
        })

conversion_manifest = {
    "schema": "rvc_conversion_manifest.v1",
    "model_name": MODEL_NAME,
    "checkpoint": str(chosen_weight),
    "checkpoint_sha256": hashlib.sha256(chosen_weight.read_bytes()).hexdigest(),
    "index": str(index_path),
    "index_sha256": hashlib.sha256(index_path.read_bytes()).hexdigest(),
    "settings": {"pitch": 0, "f0_method": F0_METHOD, "protect": 0.5, "volume_envelope": 0.8},
    "rows": conversion_rows,
}
(COMPARISON_DIR / "conversion_manifest.json").write_text(
    json.dumps(conversion_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("comparison outputs:", len(conversion_rows), "->", COMPARISON_DIR)

## 12. Compute metrics and listening sheet

In [ ]:
import numpy as np
import soundfile as sf

raw_by_probe = {row["probe_id"]: Path(row["path"]) for row in conversion_rows if row["kind"] == "raw"}
metric_rows = []
for row in conversion_rows:
    if row["kind"] != "converted":
        continue
    source_audio, source_sr = sf.read(raw_by_probe[row["probe_id"]], dtype="float32", always_2d=False)
    output_audio, output_sr = sf.read(row["path"], dtype="float32", always_2d=False)
    if output_audio.ndim != 1 or not np.isfinite(output_audio).all():
        raise RuntimeError(f"Invalid converted audio: {row['path']}")
    duration_ratio = (len(output_audio) / output_sr) / (len(source_audio) / source_sr)
    peak = float(np.max(np.abs(output_audio))) if output_audio.size else 0.0
    clipped_fraction = float(np.mean(np.abs(output_audio) >= 0.999)) if output_audio.size else 1.0
    metric_rows.append({
        **row,
        "duration_ratio": duration_ratio,
        "duration_pass": 0.95 <= duration_ratio <= 1.05,
        "peak": peak,
        "clipped_fraction": clipped_fraction,
        "clipping_pass": clipped_fraction < 0.001,
    })

metrics = {"schema": "rvc_metrics.v1", "automatic_audio": metric_rows, "cer": None, "ecapa": None}
(COMPARISON_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
listening_sheet = [
    {"probe_id": probe["id"], "identity": "", "intelligibility": "", "artifacts": "", "preferred_index_rate": ""}
    for probe in manifest["probes"]
]
(COMPARISON_DIR / "listening_sheet.json").write_text(
    json.dumps(listening_sheet, ensure_ascii=False, indent=2), encoding="utf-8"
)
for probe in manifest["probes"]:
    print("\n", probe["id"], probe["text"], "\nraw")
    display(Audio(str(raw_by_probe[probe["id"]])))
    for row in conversion_rows:
        if row["kind"] == "converted" and row["probe_id"] == probe["id"]:
            print("index", row["index_rate"])
            display(Audio(row["path"]))
print("Basic audio metrics written. The isolated CER/ECAPA cell is added by the evaluation task.")

In [ ]:
# Generated from training.rvc_eval; run it outside the Applio dependency environment.
eval_script = Path("/content/rvc_eval.py")
eval_script.write_text("\"\"\"Evaluate RVC comparison WAVs with audio, Turkish CER, and ECAPA metrics.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport statistics\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Callable, Sequence\n\nimport numpy as np\nimport soundfile as sf\nimport torch\n\nTranscriber = Callable[[Path], str]\nEmbedder = Callable[[Path], torch.Tensor]\n\n\ndef normalise_turkish(text: str) -> str:\n    \"\"\"Normalize comparison text without discarding Turkish letters.\"\"\"\n    value = re.sub(r\"[^\\w\\s]\", \"\", text.casefold(), flags=re.UNICODE)\n    return \" \".join(value.split())\n\n\ndef _edit_distance(left: str, right: str) -> int:\n    previous = list(range(len(right) + 1))\n    for left_index, left_value in enumerate(left, start=1):\n        current = [left_index]\n        for right_index, right_value in enumerate(right, start=1):\n            current.append(\n                min(\n                    current[-1] + 1,\n                    previous[right_index] + 1,\n                    previous[right_index - 1] + (left_value != right_value),\n                )\n            )\n        previous = current\n    return previous[-1]\n\n\ndef character_error_rate(reference: str, hypothesis: str) -> float:\n    reference = normalise_turkish(reference)\n    hypothesis = normalise_turkish(hypothesis)\n    if not reference:\n        return 0.0 if not hypothesis else 1.0\n    return _edit_distance(reference, hypothesis) / len(reference)\n\n\ndef _read_mono(path: Path) -> tuple[np.ndarray, int]:\n    path = Path(path)\n    if not path.is_file():\n        raise FileNotFoundError(path)\n    waveform, sample_rate = sf.read(path, dtype=\"float32\", always_2d=True)\n    if waveform.shape[1] != 1:\n        raise ValueError(f\"evaluation audio must be mono: {path}\")\n    waveform = waveform[:, 0]\n    if waveform.size == 0 or not np.isfinite(waveform).all():\n        raise ValueError(f\"evaluation audio must contain finite samples: {path}\")\n    return waveform, sample_rate\n\n\ndef audio_metrics(source_path: Path, output_path: Path) -> dict[str, float | int | bool]:\n    \"\"\"Return source-relative duration and output integrity metrics.\"\"\"\n    source, source_rate = _read_mono(Path(source_path))\n    output, output_rate = _read_mono(Path(output_path))\n    source_duration = len(source) / source_rate\n    output_duration = len(output) / output_rate\n    duration_ratio = output_duration / source_duration\n    peak = float(np.max(np.abs(output)))\n    clipped_fraction = float(np.mean(np.abs(output) >= 0.999))\n    return {\n        \"source_duration_s\": source_duration,\n        \"output_duration_s\": output_duration,\n        \"duration_ratio\": duration_ratio,\n        \"duration_pass\": 0.95 <= duration_ratio <= 1.05,\n        \"output_sample_rate\": output_rate,\n        \"output_channels\": 1,\n        \"peak\": peak,\n        \"clipped_fraction\": clipped_fraction,\n        \"clipping_pass\": clipped_fraction < 0.001,\n    }\n\n\ndef _normalized_embedding(embed: Embedder, path: Path) -> torch.Tensor:\n    value = torch.as_tensor(embed(path), dtype=torch.float32).flatten()\n    if value.numel() == 0 or not torch.isfinite(value).all() or value.norm() == 0:\n        raise ValueError(f\"invalid speaker embedding: {path}\")\n    return torch.nn.functional.normalize(value, dim=0)\n\n\ndef _mean_similarity(value: torch.Tensor, references: Sequence[torch.Tensor]) -> float:\n    return float(torch.stack([torch.dot(value, reference) for reference in references]).mean())\n\n\ndef _resolve_path(path_value: str, bundle_root: Path) -> Path:\n    path = Path(path_value)\n    return path if path.is_absolute() else bundle_root / path\n\n\ndef evaluate_rows(\n    *,\n    conversion_rows: Sequence[dict],\n    bundle_root: Path,\n    transcribe: Transcriber,\n    embed: Embedder,\n    allow_model_failure: bool = False,\n) -> list[dict]:\n    \"\"\"Evaluate converted rows against their matching raw probe and target refs.\"\"\"\n    bundle_root = Path(bundle_root)\n    bundle_manifest = json.loads(\n        (bundle_root / \"manifest.json\").read_text(encoding=\"utf-8\")\n    )\n    reference_paths = [\n        bundle_root / row[\"path\"] for row in bundle_manifest.get(\"eval_refs\", [])\n    ]\n    if not reference_paths:\n        raise ValueError(\"bundle manifest contains no eval_refs\")\n\n    raw_by_probe = {\n        row[\"probe_id\"]: _resolve_path(str(row[\"path\"]), bundle_root)\n        for row in conversion_rows\n        if row.get(\"kind\") == \"raw\"\n    }\n    reference_embeddings: list[torch.Tensor] | None\n    shared_errors: list[str] = []\n    try:\n        reference_embeddings = [\n            _normalized_embedding(embed, path) for path in reference_paths\n        ]\n    except Exception as error:\n        if not allow_model_failure:\n            raise\n        reference_embeddings = None\n        shared_errors.append(f\"ECAPA setup/reference failure: {error}\")\n\n    results: list[dict] = []\n    for row in conversion_rows:\n        if row.get(\"kind\") != \"converted\":\n            continue\n        probe_id = str(row[\"probe_id\"])\n        if probe_id not in raw_by_probe:\n            raise ValueError(f\"converted row has no raw source: {probe_id}\")\n        source_path = raw_by_probe[probe_id]\n        output_path = _resolve_path(str(row[\"path\"]), bundle_root)\n        result = {**row, **audio_metrics(source_path, output_path)}\n        errors = list(shared_errors)\n\n        try:\n            raw_hypothesis = transcribe(source_path)\n            output_hypothesis = transcribe(output_path)\n            result[\"raw_transcript\"] = raw_hypothesis\n            result[\"transcript\"] = output_hypothesis\n            result[\"raw_cer\"] = character_error_rate(str(row[\"text\"]), raw_hypothesis)\n            result[\"cer\"] = character_error_rate(str(row[\"text\"]), output_hypothesis)\n        except Exception as error:\n            if not allow_model_failure:\n                raise\n            result.update({\"raw_transcript\": None, \"transcript\": None, \"raw_cer\": None, \"cer\": None})\n            errors.append(f\"Whisper failure: {error}\")\n\n        if reference_embeddings is not None:\n            try:\n                raw_embedding = _normalized_embedding(embed, source_path)\n                output_embedding = _normalized_embedding(embed, output_path)\n                raw_similarity = _mean_similarity(raw_embedding, reference_embeddings)\n                output_similarity = _mean_similarity(output_embedding, reference_embeddings)\n                result[\"raw_speaker_similarity\"] = raw_similarity\n                result[\"speaker_similarity\"] = output_similarity\n                result[\"speaker_similarity_delta\"] = output_similarity - raw_similarity\n            except Exception as error:\n                if not allow_model_failure:\n                    raise\n                result.update(\n                    {\n                        \"raw_speaker_similarity\": None,\n                        \"speaker_similarity\": None,\n                        \"speaker_similarity_delta\": None,\n                    }\n                )\n                errors.append(f\"ECAPA failure: {error}\")\n        else:\n            result.update(\n                {\n                    \"raw_speaker_similarity\": None,\n                    \"speaker_similarity\": None,\n                    \"speaker_similarity_delta\": None,\n                }\n            )\n        result[\"evaluation_errors\"] = errors\n        results.append(result)\n    return results\n\n\ndef summarize_gates(rows: Sequence[dict]) -> dict:\n    \"\"\"Summarize automatic gates independently for every retrieval rate.\"\"\"\n    grouped: dict[float, list[dict]] = defaultdict(list)\n    for row in rows:\n        if row.get(\"index_rate\") is not None:\n            grouped[float(row[\"index_rate\"])].append(row)\n    by_rate: dict[str, dict] = {}\n    for rate in sorted(grouped):\n        values = grouped[rate]\n        gains = sum(\n            row.get(\"speaker_similarity_delta\") is not None\n            and row[\"speaker_similarity_delta\"] > 0\n            for row in values\n        )\n        cer_deltas = [\n            row[\"cer\"] - row[\"raw_cer\"]\n            for row in values\n            if row.get(\"cer\") is not None and row.get(\"raw_cer\") is not None\n        ]\n        median_cer_delta = statistics.median(cer_deltas) if cer_deltas else None\n        technical = all(\n            bool(row.get(\"duration_pass\")) and bool(row.get(\"clipping_pass\"))\n            for row in values\n        )\n        complete = len({row[\"probe_id\"] for row in values}) == 5\n        automatic_pass = bool(\n            complete\n            and gains >= 4\n            and median_cer_delta is not None\n            and median_cer_delta <= 0.05\n            and technical\n        )\n        by_rate[f\"{rate:.2f}\"] = {\n            \"probe_count\": len({row[\"probe_id\"] for row in values}),\n            \"similarity_improved_probes\": gains,\n            \"median_cer_delta\": median_cer_delta,\n            \"technical_audio_pass\": technical,\n            \"automatic_pass\": automatic_pass,\n            \"human_identity_pass\": None,\n        }\n    return {\"by_index_rate\": by_rate, \"human_review_required\": True}\n\n\ndef load_transcriber(device: str) -> Transcriber:\n    \"\"\"Load Whisper lazily so unit tests and audio-only fallback stay lightweight.\"\"\"\n    from faster_whisper import WhisperModel\n\n    model = WhisperModel(\n        \"large-v3\",\n        device=device,\n        compute_type=\"float16\" if device == \"cuda\" else \"int8\",\n    )\n\n    def transcribe(path: Path) -> str:\n        segments, _ = model.transcribe(str(path), language=\"tr\")\n        return \" \".join(segment.text for segment in segments).strip()\n\n    return transcribe\n\n\ndef load_ecapa_encoder(device: str, cache_dir: Path) -> Embedder:\n    \"\"\"Load the same ECAPA identity model used by the corpus pipeline.\"\"\"\n    import torchaudio\n    from speechbrain.pretrained import EncoderClassifier\n\n    classifier = EncoderClassifier.from_hparams(\n        source=\"speechbrain/spkrec-ecapa-voxceleb\",\n        savedir=str(cache_dir),\n        run_opts={\"device\": device},\n    )\n\n    def embed(path: Path) -> torch.Tensor:\n        waveform, sample_rate = torchaudio.load(str(path))\n        waveform = waveform.mean(dim=0, keepdim=True)\n        if sample_rate != 16_000:\n            waveform = torchaudio.functional.resample(waveform, sample_rate, 16_000)\n        with torch.inference_mode():\n            value = classifier.encode_batch(waveform.to(device)).squeeze().detach().cpu()\n        return torch.nn.functional.normalize(value, dim=0)\n\n    return embed\n\n\ndef _parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--bundle-root\", type=Path, required=True)\n    parser.add_argument(\"--conversion-manifest\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\"--cache-dir\", type=Path, default=Path(\"data/.cache/speechbrain/rvc_eval\"))\n    parser.add_argument(\"--allow-model-failure\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = _parse_args()\n    conversion = json.loads(args.conversion_manifest.read_text(encoding=\"utf-8\"))\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    setup_error: Exception | None = None\n    try:\n        transcribe = load_transcriber(device)\n        embed = load_ecapa_encoder(device, args.cache_dir)\n    except Exception as error:\n        if not args.allow_model_failure:\n            raise\n        setup_error = error\n\n        def fail(path: Path):\n            raise RuntimeError(f\"evaluation model setup failed: {setup_error}\")\n\n        transcribe = fail\n        embed = fail\n\n    rows = evaluate_rows(\n        conversion_rows=conversion[\"rows\"],\n        bundle_root=args.bundle_root,\n        transcribe=transcribe,\n        embed=embed,\n        allow_model_failure=args.allow_model_failure,\n    )\n    report = {\n        \"schema\": \"rvc_metrics.v1\",\n        \"rows\": rows,\n        \"gates\": summarize_gates(rows),\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    print(f\"metrics: {len(rows)} converted rows -> {args.output}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
eval_env = Path("/content/rvc-eval-env")
eval_python = eval_env / "bin" / "python"
full_metrics = COMPARISON_DIR / "metrics_full.json"
conversion_manifest_path = COMPARISON_DIR / "conversion_manifest.json"
try:
    if not eval_python.is_file():
        run(["uv", "venv", "--system-site-packages", str(eval_env)])
    run([
        "uv", "pip", "install", "--python", str(eval_python),
        "speechbrain<1.0", "faster-whisper>=1.2.1",
    ])
    run([
        str(eval_python), str(eval_script),
        "--bundle-root", str(BUNDLE_ROOT),
        "--conversion-manifest", str(conversion_manifest_path),
        "--output", str(full_metrics),
        "--cache-dir", "/content/rvc_eval_cache",
        "--allow-model-failure",
    ])
    print("Full CER/ECAPA report:", full_metrics)
except subprocess.CalledProcessError as error:
    fallback = {
        "error": str(error),
        "audio_metrics_preserved": str(COMPARISON_DIR / "metrics.json"),
    }
    (COMPARISON_DIR / "evaluation_setup_error.json").write_text(
        json.dumps(fallback, indent=2), encoding="utf-8"
    )
    print("LOCAL EVALUATION FALLBACK:")
    print("python -m training.rvc_eval --bundle-root <bundle> --conversion-manifest <conversion_manifest.json> --output <metrics_full.json> --allow-model-failure")

## 13. Export results ZIP

In [ ]:
training_manifest = {
    "applio_commit": APPLIO_COMMIT,
    "model_name": MODEL_NAME,
    "sample_rate": SAMPLE_RATE,
    "epochs": TOTAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "save_every_epoch": SAVE_EVERY_EPOCH,
    "f0_method": F0_METHOD,
    "embedder": EMBEDDER,
    "vocoder": VOCODER,
    "bundle_sha256": hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest(),
}
(DRIVE_OUT / "training_manifest.json").write_text(
    json.dumps(training_manifest, indent=2), encoding="utf-8"
)
shutil.copy2(BUNDLE_ROOT / "manifest.json", DRIVE_OUT / "bundle_manifest.json")
result_zip = DRIVE_OUT / "speaker_rvc_poc_v1_results.zip"
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for root in (artifact_dir, COMPARISON_DIR):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(DRIVE_OUT).as_posix())
    for name in ("training_manifest.json", "bundle_manifest.json"):
        archive.write(DRIVE_OUT / name, name)
bad_member = zipfile.ZipFile(result_zip).testzip()
assert bad_member is None, bad_member
print("RESULT READY:", result_zip, "size_GB", round(result_zip.stat().st_size / 1e9, 2))